# DTAAD

### Paper: Yu, Lu & Xue, *DTAAD*, Knowledge-Based Systems 295 (2024).
### Datasets: NASA **SMAP** and **MSL**

In [ ]:
import os, ast, math, time, copy
os.environ['TMP'] = os.environ['TEMP'] = r'D:\Personal\5th sem\ml_course\tmp_dtaad'   # temp files on D:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
from torch.func import functional_call
from scipy.stats import genpareto
from sklearn.metrics import roc_auc_score, ndcg_score
import matplotlib.pyplot as plt

DATA = 'data/SMAP_MSL' if os.path.isdir('data/SMAP_MSL') else '../data/SMAP_MSL'
os.makedirs('results', exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hyper-parameters and POT settings from paper Appendix A; low-tail probabilities are converted to upper quantiles.
WINDOW = 10            # sliding window length
LAMBDA = 0.8           # loss weight of the global (fed-back) decoder
HIDDEN = 16            # hidden units of the feed-forward layers
DROPOUT = 0.2
BATCH = 128
MAX_EPOCHS = 20
MA_WINDOW = 3          # early stopping on the moving average of the validation loss
LR = 0.01              # initial AdamW learning rate
META_TASKS = 5         # MAML (Eq. 30-31): random-batch tasks per meta update (not given in the paper)
META_ALPHA = LR        # inner-loop step size alpha
META_BETA = LR         # meta step size beta
POT_LEVEL = {'SMAP': 0.93, 'MSL': 0.99}    # Appendix low quantiles 0.07 / 0.01 -> upper-tail levels
POT_SCALE = {'SMAP': 1.00, 'MSL': 1.04}    # final multiplier on the threshold
POT_Q = 1e-4
CHANNELS = None  # populated from the channel metadata in the next section
SEEDS = [0, 1, 2, 3, 4]
print('device:', device, '| data:', DATA)

## 1. Data
Table 2 of the paper lists MSL with 58,317 train / 73,729 test points, which is exactly the total of all 27 MSL
channels, so every metadata channel of both spacecraft is used.

In [ ]:
meta = pd.read_csv(f'{DATA}/labeled_anomalies.csv')
CHANNELS = {
    ds: sorted(meta.loc[meta.spacecraft == ds, 'chan_id'].unique().tolist())
    for ds in ('SMAP', 'MSL')
}

# A channel can have multiple metadata rows (P-2 does); combine their labeled intervals.
def load_channel(chan):
    train = np.load(f'{DATA}/train/{chan}.npy')
    test = np.load(f'{DATA}/test/{chan}.npy')
    labels = np.zeros(len(test), dtype=int)
    rows = meta[meta.chan_id == chan]
    if rows.empty:
        raise KeyError(f'No metadata found for channel {chan}')
    for value in rows.anomaly_sequences:
        for start, end in ast.literal_eval(value):
            labels[start:end + 1] = 1
    return train, test, labels

for ds, channels in CHANNELS.items():
    shapes = sorted({load_channel(chan)[0].shape[1] for chan in channels})
    print(f'{ds}: {len(channels)} unique channels; feature counts {shapes}')

# Plot one representative channel per spacecraft; experiments below use every channel.
fig, axes = plt.subplots(2, 1, figsize=(12, 4))
for ax, ds in zip(axes, ('SMAP', 'MSL')):
    chan = 'P-1' if ds == 'SMAP' else 'C-1'
    train, test, labels = load_channel(chan)
    ax.plot(np.arange(len(train)), train[:, 0], lw=0.6, color='gray', label='train')
    ax.plot(np.arange(len(train), len(train) + len(test)), test[:, 0], lw=0.6, color='black', label='test')
    ax.fill_between(np.arange(len(train), len(train) + len(test)), -1, 1, where=labels == 1,
                    color='red', alpha=0.2, label='anomaly')
    ax.set_title(f'{ds} {chan}: telemetry (dim 0)'); ax.legend(loc='upper left', fontsize=7)
plt.tight_layout(); plt.savefig('results/1_data.png', dpi=150); plt.show()


## 2. Preprocessing (Section 3.2 of the paper)
1. **Min-max scaling** with the training min/max (Eq. 3). A column that never changes in training is divided by
   1e-4, so any change in it at test time becomes a very large value. This turns out to be the main anomaly signal
   on some channels: a command that never occurred in training.
2. **White-noise augmentation** at SNR = 50 dB, added as ε/100 (Eq. 4–5). Training data only.
3. **Sliding windows** of length 10 ending at the current step, padded at the start by repeating x₁.

In [ ]:
def preprocess(train, test, seed=0):
    lo, hi = train.min(0), train.max(0)
    train = (train - lo) / (hi - lo + 1e-4)                              # Eq. 3
    test = (test - lo) / (hi - lo + 1e-4)
    noise_power = np.mean(train ** 2) / 10 ** (50 / 10)                 # Eq. 4: aggregate signal power, SNR = 50 dB
    noise = np.random.default_rng(seed).normal(size=train.shape) * np.sqrt(noise_power)
    return train + noise / 100, test                                     # Eq. 5

def make_windows(x):
    padded = np.concatenate([np.repeat(x[:1], WINDOW - 1, axis=0), x])
    windows = np.stack([padded[t:t + WINDOW] for t in range(len(x))])   # (time, window, dims)
    return torch.tensor(windows, dtype=torch.float32, device=device)

train_raw, test_raw, _ = load_channel('P-1')  # representative SMAP channel
train_n, test_n = preprocess(train_raw, test_raw)
print(f'P-1 normalised train range [{train_n.min():.3f}, {train_n.max():.3f}], test max {test_n.max():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(train_raw[:500, 0], color='gray', label='raw'); axes[0].plot(train_n[:500, 0], color='black', label='normalised + noise')
axes[0].set_title('P-1 dim 0 (first 500 steps)'); axes[0].legend(fontsize=7)
axes[1].imshow(train_n.T, aspect='auto', interpolation='nearest', cmap='gray_r')
axes[1].set(title='P-1 normalised train, all dims', xlabel='t', ylabel='dim')
plt.tight_layout(); plt.savefig('results/2_preprocessing.png', dpi=150); plt.show()

## 3. Model
* **Local branch:** causal TCN (3 layers, kernel 4) → Transformer encoder → decoder, giving prediction O₂.
* **Feedback:** O₂ is added to every step of the window.
* **Global branch:** dilated causal TCN over time (dilations 1, 2, 4, kernel 3; layer count from Eq. 20, Fig. 3) → encoder → decoder, giving the final prediction Ô₁.
* Encoder: positional encoding, multi-head attention with heads = number of dims, FFN, post-LayerNorm.

In [ ]:
class TCNLayer(nn.Module):
    def __init__(self, channels, kernel, dilation):
        super().__init__()
        self.pad = (kernel - 1) * dilation
        self.conv = weight_norm(nn.Conv1d(channels, channels, kernel, dilation=dilation))
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):                                   # x: (batch, dims, time)
        return self.drop(F.leaky_relu(self.conv(F.pad(x, (self.pad, 0))), 0.01))

def make_tcn(dims, kind):
    if kind == 'local':
        return nn.Sequential(*[TCNLayer(dims, 4, 1) for _ in range(3)])
    n_layers = math.ceil(math.log2((WINDOW - 1) / (3 - 1) + 1))          # Eq. 20 -> 3 layers
    return nn.Sequential(*[TCNLayer(dims, 3, 2 ** i) for i in range(n_layers)])   # dilations 1, 2, 4 (Fig. 3)

def ffn(d_in, d_out):                                                    # Eq. 24
    return nn.Sequential(nn.Linear(d_in, HIDDEN), nn.LeakyReLU(0.01), nn.Linear(HIDDEN, d_out))

def positional_encoding(length, dims):
    pos = torch.arange(length).unsqueeze(1)
    freq = torch.exp(torch.arange(0, dims, 2) * (-math.log(10000.0) / dims))
    pe = torch.zeros(length, dims)
    pe[:, 0::2] = torch.sin(pos * freq)
    pe[:, 1::2] = torch.cos(pos * freq)[:, :dims // 2]
    return pe

class Branch(nn.Module):
    """TCN -> Transformer encoder -> decoder. Input (batch, window, dims), output (batch, dims)."""
    def __init__(self, dims, kind, use_transformer=True):
        super().__init__()
        self.tcn = make_tcn(dims, kind)
        self.use_transformer = use_transformer
        self.register_buffer('pe', positional_encoding(WINDOW, dims))
        self.attention = nn.MultiheadAttention(dims, num_heads=dims, dropout=DROPOUT, batch_first=True)
        self.norm1, self.norm2 = nn.LayerNorm(dims), nn.LayerNorm(dims)
        self.encoder_ffn, self.residual_ffn = ffn(dims, dims), ffn(dims, dims)
        self.decoder = ffn(WINDOW, 1)
        self.drop = nn.Dropout(DROPOUT)

    def encode(self, x):
        h = self.tcn(x.transpose(1, 2)).transpose(1, 2)                  # both TCNs convolve over time
        if self.use_transformer:                                         # Eq. 25
            h = h * math.sqrt(h.shape[-1]) + self.pe
            h = self.norm1(h + self.drop(self.attention(h, h, h)[0]))
            h = self.norm2(h + self.drop(self.encoder_ffn(h)))
        h = h + self.residual_ffn(h)                                     # Eq. 26
        return h.transpose(1, 2)                                         # (batch, dims, window)

    def forward(self, x):
        return torch.sigmoid(self.decoder(self.encode(x))).squeeze(-1)

class DTAAD(nn.Module):
    def __init__(self, dims, local=True, glob=True, feedback=True, transformer=True):
        super().__init__()
        self.local = Branch(dims, 'local', transformer) if local else None
        self.glob = Branch(dims, 'global', transformer) if glob else None
        self.feedback = feedback

    def forward(self, x):
        o2 = self.local(x) if self.local else None                       # local prediction
        if self.glob is None:
            return None, o2
        if self.feedback and o2 is not None:
            x = x + o2.unsqueeze(1)                                      # feed O2 back over the whole window
        return self.glob(x), o2                                          # global prediction Ô1

print('parameters (25 dims):', sum(p.numel() for p in DTAAD(25).parameters()))

## 4. Training
* Loss (Eq. 28): 0.8·MSE(Ô₁, xₜ) + 0.2·MSE(O₂, xₜ). AdamW (lr 0.01, weight decay 1e-5), learning rate ×0.9 every 5 epochs.
* **Meta-learning (Eq. 30–31, Algorithm 1 line 13):** after each epoch, one MAML update. For each of a few random-batch
  tasks, θ′ = θ − α∇L(support batch); then θ ← θ − β∇Σ L_{θ′}(query batch).
* **Early stopping (Appendix A):** stop once the moving average of the validation loss starts to rise; keep the best weights.

In [ ]:
def loss_fn(model, x, params=None):
    out = model(x) if params is None else functional_call(model, params, (x,))
    target = x[:, -1]                                                    # the current step x_t
    if isinstance(out, dict): return fusion_loss(out, target)            # AdaptiveDTAAD (Section 10)
    o1, o2 = out
    if o1 is None: return F.mse_loss(o2, target)
    if o2 is None: return F.mse_loss(o1, target)
    return LAMBDA * F.mse_loss(o1, target) + (1 - LAMBDA) * F.mse_loss(o2, target)

def meta_step(model, train_w):
    # MAML (Eq. 30-31): adapt θ on one random batch per task, then update θ with the gradient of the
    # adapted models' losses on a second random batch. Ablated components give no gradient (None).
    names, params = zip(*model.named_parameters())
    meta_loss = 0
    for _ in range(META_TASKS):
        support, query = torch.randperm(len(train_w))[:2 * BATCH].chunk(2)
        grads = torch.autograd.grad(loss_fn(model, train_w[support]), params, create_graph=True, allow_unused=True)
        adapted = {n: p if g is None else p - META_ALPHA * g for n, p, g in zip(names, params, grads)}   # Eq. 30
        meta_loss = meta_loss + loss_fn(model, train_w[query], adapted)
    grads = torch.autograd.grad(meta_loss, params, allow_unused=True)
    with torch.no_grad():
        for p, g in zip(params, grads):
            if g is not None:
                p -= META_BETA * g                                                                     # Eq. 31

def train_model(model, train_w, val_w, lr):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.9)
    history, best, best_state = [], float('inf'), None
    for epoch in range(MAX_EPOCHS):
        start = time.time()
        model.train()
        losses = []
        for idx in torch.randperm(len(train_w)).split(BATCH):
            loss = loss_fn(model, train_w[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        meta_step(model, train_w)                                        # Algorithm 1, line 13
        sched.step()
        model.eval()
        with torch.no_grad():
            val = loss_fn(model, val_w).item()
        history.append({'epoch': epoch + 1, 'train': np.mean(losses), 'val': val, 'seconds': time.time() - start})
        if val < best:
            best, best_state = val, copy.deepcopy(model.state_dict())
        vals = [h['val'] for h in history]                               # stop when the moving average rises
        if len(vals) > MA_WINDOW and np.mean(vals[-MA_WINDOW:]) > np.mean(vals[-MA_WINDOW - 1:-1]):
            break
    model.load_state_dict(best_state)
    return pd.DataFrame(history)

## 5. Score, threshold and metrics
* **Score (Eq. 11):** squared error between xₜ and Ô₁ for every dimension.
* **Detection (Section 4.3, Eq. 6, Algorithm 2):** one POT threshold per dimension, fitted on that dimension's training
  scores; a time step is anomalous if any dimension exceeds its threshold, D = ∨ᵢ yᵢ.
* For comparison, `evaluate` also applies a single POT to the scores averaged over dimensions, which is what the
  TranAD/DTAAD reference code reports.
* Point adjustment, then P, R, F1 (Eq. 34–36) and AUC.

In [ ]:
@torch.no_grad()
def anomaly_scores(model, windows):
    model.eval()
    o1, o2 = model(windows)
    pred = o1 if o1 is not None else o2
    return ((pred - windows[:, -1]) ** 2).cpu().numpy()

def pot_threshold(train_score, test_score, ds, q=POT_Q):
    level = POT_LEVEL[ds]
    while True:                                          # lower the level until the fit works
        t = np.sort(train_score)[int(level * len(train_score))]
        peaks = train_score[train_score > t] - t
        if len(peaks) >= 2 or level < 0.5:
            break
        level *= 0.999
    if len(peaks) < 2 or peaks.std() == 0:               # degenerate case: (almost) constant scores
        return train_score.max() * POT_SCALE[ds]
    gamma, _, beta = genpareto.fit(peaks, floc=0)        # maximum likelihood
    r = q * len(train_score) / len(peaks)
    z = t - beta * np.log(r) if abs(gamma) < 1e-8 else t + beta / gamma * (r ** (-gamma) - 1)
    above = np.where(test_score > t)[0]
    first = above[0] if len(above) else len(test_score)  # from here on SPOT uses t instead of z
    thresholds = np.r_[np.full(first, z), np.full(len(test_score) - first, t)]
    return thresholds.mean() * POT_SCALE[ds]

def point_adjust(pred, labels):
    pred = pred.copy()
    edges = np.diff(np.r_[0, labels, 0])
    for start, end in zip(np.where(edges == 1)[0], np.where(edges == -1)[0]):
        if pred[start:end].any():
            pred[start:end] = True
    return pred

def f1_score(pred, labels):
    tp = (pred & (labels == 1)).sum()
    precision, recall = tp / max(pred.sum(), 1), tp / labels.sum()
    return precision, recall, 2 * precision * recall / max(precision + recall, 1e-12)

def evaluate(train_scores, test_scores, labels, ds):
    train_score, test_score = train_scores.mean(axis=1), test_scores.mean(axis=1)   # mean over dimensions
    threshold = pot_threshold(train_score, test_score, ds)
    pred = test_score >= threshold
    adjusted = point_adjust(pred, labels)
    p, r, f1 = f1_score(adjusted, labels)
    metrics = {'P': p, 'R': r, 'F1': f1, 'AUC': roc_auc_score(labels, adjusted),
               'raw F1': f1_score(pred, labels)[2]}
    return metrics, threshold, adjusted, pred

def evaluate_dimensions(train_scores, test_scores, labels, ds, q=POT_Q):
    # Scores are (time, features). Fit one independent POT threshold per feature.
    # NASA labels are channel-level timestamps, so each feature uses the same labels.
    if train_scores.ndim != 2 or test_scores.ndim != 2:
        raise ValueError('Expected train/test scores shaped (time, features)')
    if train_scores.shape[1] != test_scores.shape[1] or len(test_scores) != len(labels):
        raise ValueError('Score feature counts or test/label lengths do not match')

    per_dimension = []
    adjusted_columns, raw_columns = [], []
    for dim in range(train_scores.shape[1]):
        threshold = pot_threshold(train_scores[:, dim], test_scores[:, dim], ds, q)
        raw_pred = test_scores[:, dim] >= threshold
        adjusted = point_adjust(raw_pred, labels)
        precision, recall, f1 = f1_score(adjusted, labels)
        per_dimension.append({
            'dimension': dim, 'P': precision, 'R': recall, 'F1': f1,
            'AUC': roc_auc_score(labels, adjusted),
            'raw F1': f1_score(raw_pred, labels)[2], 'threshold': threshold
        })
        adjusted_columns.append(adjusted)
        raw_columns.append(raw_pred)

    adjusted_by_dim = np.column_stack(adjusted_columns)
    raw_by_dim = np.column_stack(raw_columns)
    adjusted_union = adjusted_by_dim.any(axis=1)
    raw_union = raw_by_dim.any(axis=1)
    precision, recall, f1 = f1_score(adjusted_union, labels)
    union_metrics = {
        'P': precision, 'R': recall, 'F1': f1,
        'AUC': roc_auc_score(labels, adjusted_union),
        'raw F1': f1_score(raw_union, labels)[2]
    }
    return per_dimension, union_metrics, adjusted_by_dim, raw_by_dim

## 6. Running an experiment

In [ ]:
def prepare(chan, train_fraction=1.0, seed=0):
    train, test, labels = load_channel(chan)
    if train_fraction < 1:                                    # "20% data" setting: middle slice of the series
        mid, half = len(train) // 2, int(len(train) * train_fraction / 2)
        train = train[mid - half: mid + half]
    train, test = preprocess(train, test, seed)
    return make_windows(train), make_windows(test), labels

def run(ds, chan, train_fraction=1.0, seed=0, **variant):
    torch.manual_seed(seed); np.random.seed(seed)
    train_w, test_w, labels = prepare(chan, train_fraction, seed)
    n_val = len(train_w) // 5
    model = DTAAD(train_w.shape[2], **variant).to(device)
    history = train_model(model, train_w[:-n_val], train_w[-n_val:], LR)
    train_scores, test_scores = anomaly_scores(model, train_w), anomaly_scores(model, test_w)
    mean_metrics, threshold, mean_adjusted, mean_raw_pred = evaluate(train_scores, test_scores, labels, ds)
    dimension_metrics, metrics, dim_adjusted, dim_raw_pred = evaluate_dimensions(   # Eq. 6: OR over dimensions
        train_scores, test_scores, labels, ds)
    metrics['epochs'], metrics['s/epoch'] = len(history), history.seconds.mean()
    return dict(metrics=metrics, mean_metrics=mean_metrics, history=history, test_score=test_scores.mean(axis=1),
                threshold=threshold, labels=labels, channel=chan, dimension_metrics=dimension_metrics, model=model,
                adjusted=dim_adjusted.any(axis=1), raw_pred=dim_raw_pred.any(axis=1),
                mean_adjusted=mean_adjusted, mean_raw_pred=mean_raw_pred)

## 7. Main results (all metadata channels, 100% and 20% training data, 5 seeds)

Each channel gets its own model and per-dimension POT thresholds. Dataset-level precision, recall, and F1 pool the point predictions and labels across that dataset's channels for each seed; the table then reports the mean and standard deviation across seeds. `F1 (mean-score POT)` is the single-threshold variant for comparison. `per_channel_results.csv` keeps the channel-level scores for inspection.

In [ ]:
per_channel_runs = {}
for ds, channels in CHANNELS.items():
    for frac in (1.0, 0.2):
        for chan in channels:
            per_channel_runs[(ds, frac, chan)] = [
                run(ds, chan, frac, seed) for seed in SEEDS
            ]

paper = {('SMAP', 1.0): (0.902, 0.991), ('SMAP', 0.2): (0.900, 0.989),
         ('MSL', 1.0): (0.950, 0.992), ('MSL', 0.2): (0.921, 0.986)}

def pooled_metrics(channel_results, seed_index):
    def pool(key): return np.concatenate([rs[seed_index][key] for rs in channel_results])
    labels, adjusted = pool('labels'), pool('adjusted')
    p, r, f1 = f1_score(adjusted, labels)
    return {'P': p, 'R': r, 'F1': f1, 'AUC': roc_auc_score(labels, adjusted),
            'raw F1': f1_score(pool('raw_pred'), labels)[2],
            'mean-score F1': f1_score(pool('mean_adjusted'), labels)[2]}

rows, channel_rows, dimension_rows = [], [], []
for ds, channels in CHANNELS.items():
    for frac in (1.0, 0.2):
        channel_results = [per_channel_runs[(ds, frac, chan)] for chan in channels]
        seed_metrics = pd.DataFrame([pooled_metrics(channel_results, i) for i in range(len(SEEDS))])
        epoch_times = [r['metrics']['s/epoch'] for runs_for_channel in channel_results for r in runs_for_channel]
        rows.append({'dataset': ds, 'channels': len(channels), 'train data': f'{frac:.0%}',
                     'P': f"{seed_metrics.P.mean():.3f}", 'R': f"{seed_metrics.R.mean():.3f}",
                     'F1 (ours)': f"{seed_metrics.F1.mean():.3f} +/- {seed_metrics.F1.std():.3f}",
                     'F1 (mean-score POT)': f"{seed_metrics['mean-score F1'].mean():.3f} +/- {seed_metrics['mean-score F1'].std():.3f}",
                     'F1 (paper)': paper[ds, frac][0], 'AUC (ours)': f"{seed_metrics.AUC.mean():.3f}",
                     'AUC (paper)': paper[ds, frac][1], 'raw F1': f"{seed_metrics['raw F1'].mean():.3f}",
                     's/epoch': f'{np.mean(epoch_times):.2f}'})
        for chan in channels:
            for seed, result in zip(SEEDS, per_channel_runs[(ds, frac, chan)]):
                channel_rows.append({'dataset': ds, 'channel': chan, 'train data': f'{frac:.0%}',
                                     'seed': seed, **result['metrics'],
                                     **{f'mean-score {k}': v for k, v in result['mean_metrics'].items()}})
                for dim_result in result['dimension_metrics']:
                    dimension_rows.append({'dataset': ds, 'channel': chan, 'train data': f'{frac:.0%}',
                                           'seed': seed, **dim_result})
table = pd.DataFrame(rows)
per_channel_table = pd.DataFrame(channel_rows)
per_dimension_table = pd.DataFrame(dimension_rows)
print(table.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
for ax, ds in zip(axes, ('SMAP', 'MSL')):
    chan = 'P-1' if ds == 'SMAP' else 'C-1'
    h = per_channel_runs[(ds, 1.0, chan)][0]['history']
    ax.plot(h.epoch, h.train, color='black', label='train loss')
    ax.plot(h.epoch, h.val, color='gray', ls='--', label='validation loss')
    ax.set(yscale='log', xlabel='epoch', ylabel='MSE', title=f'{ds} {chan}: loss curves (seed 0)'); ax.legend()
plt.tight_layout(); plt.savefig('results/3_loss_curves.png', dpi=150); plt.show()


## 8. Ablation study (all metadata channels, 20% training data, 5 seeds)

Ablation values are pooled across channels per seed, using each channel's own learned model and threshold.


In [ ]:
variants = {'DTAAD': {}, 'w/o local TCN': dict(local=False), 'w/o global TCN': dict(glob=False),
            'w/o feedback': dict(feedback=False), 'w/o Transformer': dict(transformer=False)}
paper_t7 = {'SMAP': [(0.9892, 0.8985), (0.9869, 0.8805), (0.9870, 0.8815), (0.9854, 0.8688), (0.9853, 0.8682)],
            'MSL': [(0.9864, 0.9212), (0.9196, 0.6638), (0.9316, 0.6989), (0.9688, 0.8356), (0.9700, 0.8412)]}
rows = []
for ds, channels in CHANNELS.items():
    for (name, flags), (paper_auc, paper_f1) in zip(variants.items(), paper_t7[ds]):
        seed_metrics = []
        for seed in SEEDS:
            if name == 'DTAAD':
                seed_metrics.append(pooled_metrics([per_channel_runs[(ds, 0.2, chan)] for chan in channels], seed))
            else:
                seed_metrics.append(pooled_metrics([[run(ds, chan, 0.2, seed, **flags)] for chan in channels], 0))
        seed_metrics = pd.DataFrame(seed_metrics)
        rows.append({'dataset': ds, 'channels': len(channels), 'variant': name,
                     'AUC* (ours)': f'{seed_metrics.AUC.mean():.3f}', 'AUC* (paper)': paper_auc,
                     'F1* (ours)': f'{seed_metrics.F1.mean():.3f} +/- {seed_metrics.F1.std():.3f}', 'F1* (paper)': paper_f1})
ablation = pd.DataFrame(rows)
print(ablation.to_string(index=False))

In [ ]:
table.to_csv('results/main_results.csv', index=False); per_channel_table.to_csv('results/per_channel_results.csv', index=False); per_dimension_table.to_csv('results/per_dimension_results.csv', index=False); ablation.to_csv('results/ablation.csv', index=False)


## 9. Novelty 1: counterfactual sensor attribution

The paper diagnoses an anomaly by ranking sensors by their own anomaly score (Table 5, HitRate and NDCG). A sensor with
a large score is not necessarily the one that caused the anomaly: a fault in one sensor spreads to the sensors that
depend on it, and those can deviate even more.

**Counterfactual attribution.** At an anomalous time step $t$, for each sensor $c$, replace $c$'s values in the input
window with the model's own reconstruction of $c$ and score again:

$$I_c = S(X) - S(X_{\setminus c}), \qquad S = \text{mean squared error of } \hat O_1 \text{ over sensors}.$$

A large positive $I_c$ means sensor $c$ accounts for much of the anomaly score, directly and through the other sensors
the model predicts from it.

**Dependency-aware ranking.** A directed dependency matrix $A$ is estimated from normal training data: the strongest
lagged cross-correlation from $c$'s past to $d$'s present, keeping only the stronger direction of each pair, with rows
normalised. Credit then flows upstream: $R_c = I_c + \sum_d A_{cd} I_d$.

**Evaluation.** Diagnosis is scored at the ground-truth anomalous time steps, as in the paper's HitRate/NDCG protocol.
* *Synthetic system with a known cause.* A linear system with lagged edges 0→2→4, 1→3→5 and {0, 1}→6 (sensor 7 is
  independent). A fault (spike, level shift or drift) enters one root sensor and spreads downstream. Edges 2→4 and 3→5
  have gain 1.4, so a descendant can deviate more than the root. Metrics: root-cause top-1/top-3 accuracy, and
  HitRate@100/150% and NDCG@100/150% against the root plus its descendants.
* *Faults injected into SMAP/MSL.* NASA labels are per channel, with no per-sensor ground truth, so single-sensor
  faults are injected into normal test regions of every channel. Each channel is diagnosed with its own model (100%
  data, first seed) from Section 7.
* *Cost.* Wall-clock seconds per diagnosed time step, excluding the forward pass that detection already needs.

Counterfactual attribution identifies **influential** sensors, not causal root causes. Establishing causality needs
stronger assumptions or interventions.

In [ ]:
@torch.no_grad()
def sensor_errors(model, windows, chunk=4096):
    # Per-sensor squared error of the model's final prediction (Ô1, or the fused mean of AdaptiveDTAAD).
    model.eval()
    errors, preds = [], []
    for w in windows.split(chunk):
        out = model(w)
        pred = out['mu'] if isinstance(out, dict) else (out[0] if out[0] is not None else out[1])
        errors.append((pred - w[:, -1]) ** 2); preds.append(pred)
    return torch.cat(errors), torch.cat(preds)

@torch.no_grad()
def counterfactual_attribution(model, windows, times, errors, pred, chunk=64):
    # I[t, c] = S(X) - S(X without c): sensor c's whole window column is replaced by the model's reconstruction of it.
    recon_w = make_windows(pred.cpu().numpy())               # reconstructions, windowed like the input
    m = windows.shape[2]
    replace = torch.eye(m, dtype=torch.bool, device=windows.device)[None, :, None, :]   # copy c has column c replaced
    I = []
    for idx in times.split(chunk):
        x_cf = torch.where(replace, recon_w[idx][:, None], windows[idx][:, None])      # (n, c, window, dims)
        e_cf = sensor_errors(model, x_cf.reshape(-1, WINDOW, m))[0].reshape(len(idx), m, m)
        I.append(errors[idx].mean(-1, keepdim=True) - e_cf.mean(-1))
    return torch.cat(I).cpu().numpy()

def lagged_dependency(train, max_lag=WINDOW - 1):
    # A[c, d] > 0 when c's past tracks d's present more strongly than the reverse; rows sum to 1.
    z = (train - train.mean(0)) / (train.std(0) + 1e-8)
    C = np.zeros((z.shape[1], z.shape[1]))
    for lag in range(1, max_lag + 1):
        C = np.maximum(C, np.abs(z[:-lag].T @ z[lag:]) / (len(z) - lag))     # C[c, d] = |corr(x_c[t-lag], x_d[t])|
    A = np.clip(C - C.T, 0, None)
    return A / (A.sum(1, keepdims=True) + 1e-8)

def diagnose(model, windows, times, A):
    # Per-sensor rankings at `times` for each method, and seconds per diagnosed step.
    times = torch.as_tensor(times)
    errors, pred = sensor_errors(model, windows)             # the detection pass, shared by all methods
    rankings = {'score': errors[times].cpu().numpy()}       # the paper's diagnosis
    start = time.time()
    I = counterfactual_attribution(model, windows, times, errors, pred)
    cost = (time.time() - start) / len(times)
    rankings['counterfactual'], rankings['counterfactual + dependency'] = I, I + I @ A.T
    return rankings, {'score': 0.0, 'counterfactual': cost, 'counterfactual + dependency': cost}

def diagnosis_metrics(scores, truth, roots):
    # HitRate@P% and NDCG@P% as in the paper (Table 5), plus top-1/top-3 root-cause accuracy.
    order = np.argsort(-scores, axis=1)
    out = {}
    for p in (1.0, 1.5):
        k = np.rint(p * truth.sum(1)).astype(int)
        out[f'H@{p:.0%}'] = np.mean([truth[i, order[i, :k[i]]].sum() / truth[i].sum() for i in range(len(truth))])
        out[f'N@{p:.0%}'] = sum(ndcg_score(truth[k == kk], scores[k == kk], k=kk) * (k == kk).sum()
                                for kk in np.unique(k)) / len(k)
    out['root top-1'] = np.mean(order[:, 0] == roots)
    out['root top-3'] = np.mean((order[:, :3] == roots[:, None]).any(1))
    return out

def diagnosis_table(runs):
    # runs: list of ((rankings, cost), truth, roots). Pools all diagnosed steps per method.
    truth, roots = np.concatenate([r[1] for r in runs]), np.concatenate([r[2] for r in runs])
    return pd.DataFrame([{'method': method,
                          **diagnosis_metrics(np.concatenate([r[0][0][method] for r in runs]), truth, roots),
                          's / step': np.mean([r[0][1][method] for r in runs])}
                         for method in runs[0][0][0]])

In [ ]:
SYN_PARENTS = {2: [(0, 2, 0.9)], 3: [(1, 1, 0.9)], 4: [(2, 1, 1.4)], 5: [(3, 2, 1.4)],
               6: [(0, 3, 0.6), (1, 1, 0.6)]}             # child: [(parent, lag, gain)], in topological order
SYN_SOURCES = {0: 50, 1: 80, 7: 30}                        # source sensor: period of its sinusoid
SYN_ROOTS = [0, 1, 2, 3]                                   # fault roots: sensors with descendants
SYN_FAULT_SIZE = {'spike': 1.0, 'shift': 0.5, 'drift': 0.8}

def descendants(c):
    children = [k for k, parents in SYN_PARENTS.items() if any(p == c for p, _, _ in parents)]
    return set(children).union(*(descendants(k) for k in children))

def synthetic_system(length, rng, faults=()):
    # A fault (start, end, root, kind) enters the root's innovation and spreads along the lagged edges.
    t = np.arange(length)
    x = rng.normal(scale=0.05, size=(length, 8))
    for start, end, root, kind in faults:
        x[start:end, root] += SYN_FAULT_SIZE[kind] * (np.linspace(0, 1, end - start) if kind == 'drift' else 1)
    for c, period in SYN_SOURCES.items():
        x[:, c] += np.sin(2 * np.pi * t / period + c)
    for c, parents in SYN_PARENTS.items():
        for p, lag, gain in parents:
            x[lag:, c] += gain * x[:-lag, p]
    return x

def synthetic_benchmark(seed, events=35, length=6000):
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    faults = []
    for start in np.linspace(150, length - 150, events).astype(int):
        kind = str(rng.choice(list(SYN_FAULT_SIZE)))
        end = start + (3 if kind == 'spike' else int(rng.integers(10, 41)))
        faults.append((start, end, int(rng.choice(SYN_ROOTS)), kind))
    train, test = preprocess(synthetic_system(5000, rng), synthetic_system(length, rng, faults), seed)
    train_w, test_w = make_windows(train), make_windows(test)
    n_val = len(train_w) // 5
    model = DTAAD(train.shape[1]).to(device)
    train_model(model, train_w[:-n_val], train_w[-n_val:], LR)
    truth, roots = np.zeros(test.shape, dtype=int), np.full(len(test), -1)
    for start, end, root, _ in faults:
        span = slice(start, end + 3)                       # descendants react up to 3 steps later
        truth[span, [root, *descendants(root)]] = 1
        roots[span] = root
    times = np.where(roots >= 0)[0]
    return diagnose(model, test_w, times, lagged_dependency(train)), truth[times], roots[times]

INJECT_SIZE = {'spike': 0.5, 'shift': 0.3}

def injection_benchmark(ds, chan, rng, events=10):
    # Inject single-sensor faults into normal test regions and diagnose them with the channel's Section 7 model.
    train_w, test_w, labels = prepare(chan, 1.0, SEEDS[0])
    train, test = train_w[:, -1].cpu().numpy(), test_w[:, -1].cpu().numpy().copy()
    varying = np.where(train.std(0) > 1e-3)[0]
    if len(varying) < 2:
        return None
    normal = np.convolve(labels, np.ones(2 * WINDOW + 50), 'same') == 0       # no labelled anomaly nearby
    candidates = np.where(normal)[0]
    candidates = candidates[(candidates > WINDOW) & (candidates < len(test) - 40)]
    truth, roots, last = np.zeros(test.shape, dtype=int), np.full(len(test), -1), -np.inf
    for start in np.sort(rng.choice(candidates, size=min(events, len(candidates)), replace=False)):
        kind, sensor = str(rng.choice(list(INJECT_SIZE))), int(rng.choice(varying))
        end = start + (3 if kind == 'spike' else int(rng.integers(10, 31)))
        if start - last < 100 or not normal[end]:
            continue
        test[start:end, sensor] += INJECT_SIZE[kind]
        truth[start:end, sensor], roots[start:end], last = 1, sensor, start
    times = np.where(roots >= 0)[0]
    if len(times) == 0:
        return None
    model = per_channel_runs[(ds, 1.0, chan)][0]['model']
    return diagnose(model, make_windows(test), times, lagged_dependency(train)), truth[times], roots[times]

In [ ]:
rows = [{'benchmark': 'synthetic (known root)', **row}
        for row in diagnosis_table([synthetic_benchmark(seed) for seed in SEEDS]).to_dict('records')]
rng = np.random.default_rng(0)
for ds, channels in CHANNELS.items():
    runs = [r for r in (injection_benchmark(ds, chan, rng) for chan in channels) if r is not None]
    rows += [{'benchmark': f'{ds} injected ({len(runs)} channels)', **row} for row in diagnosis_table(runs).to_dict('records')]
diagnosis = pd.DataFrame(rows)
print(diagnosis.round(4).to_string(index=False))
diagnosis.to_csv('results/diagnosis.csv', index=False)

## 10. Novelty 2: adaptive, uncertainty-aware local–global fusion

DTAAD combines its two branches only in the loss, with a fixed $\lambda = 0.8$, and detects with the global
prediction $\hat O_1$ alone. Point anomalies are short and local, while contextual ones change longer patterns, so
which branch is more reliable can change from step to step and from sensor to sensor.

* **Uncertainty per branch.** Each branch gets a second decoder head that predicts a log-variance per sensor,
  $\log\sigma^2$. Training uses a β-NLL Gaussian loss (β = 0.5) on the fused output plus each branch, with the
  paper's λ between the branches.
* **Dynamic branch weighting.** The branches are fused by precision weighting:
  $\mu = w\mu_1 + (1-w)\mu_2$ with $w = \sigma_2^2 / (\sigma_1^2 + \sigma_2^2)$, and $\sigma^2 = (\sigma_1^{-2} + \sigma_2^{-2})^{-1}$.
  The more confident branch dominates, per time step and per sensor, with no extra parameters.
* **Sensor-wise uncertainty-aware score.** $s_{t,c} = (x_{t,c} - \mu_{t,c})^2 / \sigma^2_{t,c}$. A residual counts more
  on a sensor the model is usually sure about.
* **Per-sensor POT calibration.** Each sensor's POT is fitted on held-out validation scores rather than training
  scores, and the risk $q$ is split across the $m$ sensors ($q/m$). Otherwise the OR in Eq. 6 multiplies false alarms
  by the number of sensors.

Variants, each adding one step: DTAAD (paper) → DTAAD + calibrated POT → adaptive fusion (squared error of the fused
mean) → + uncertainty score → + calibrated POT. The DTAAD models are the ones from Section 7 (100% data).

**Research question:** can adaptive fusion improve detection across anomalies with different temporal
characteristics? The NASA metadata labels each anomaly segment *point* or *contextual*. For each type and length
(short < 100 steps, long ≥ 100) we report the fraction of segments detected and the median detection delay (raw
predictions, before point adjustment). We also report the gate $w$ (weight of the global branch) inside each anomaly
type and in normal periods.

In [ ]:
BETA_NLL = 0.5

class UncertainBranch(Branch):
    # Branch with a second decoder head for each sensor's log-variance.
    def __init__(self, dims, kind):
        super().__init__(dims, kind)
        self.logvar_decoder = ffn(WINDOW, 1)

    def forward(self, x):
        h = self.encode(x)
        mean = torch.sigmoid(self.decoder(h)).squeeze(-1)
        logvar = 8 * torch.tanh(self.logvar_decoder(h).squeeze(-1) / 8) - 4      # bounded to (-12, 4)
        return mean, logvar

class AdaptiveDTAAD(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.local, self.glob = UncertainBranch(dims, 'local'), UncertainBranch(dims, 'global')

    def forward(self, x):
        mu2, lv2 = self.local(x)
        mu1, lv1 = self.glob(x + mu2.unsqueeze(1))                       # same feedback as DTAAD
        w = torch.sigmoid(lv2 - lv1)                                     # = σ2² / (σ1² + σ2²): weight of the global branch
        return dict(mu=w * mu1 + (1 - w) * mu2, logvar=-torch.logaddexp(-lv1, -lv2), w=w,
                    mu1=mu1, lv1=lv1, mu2=mu2, lv2=lv2)

def gaussian_nll(mean, logvar, target):
    # β-NLL (Seitzer et al., 2022): weighting by σ^(2β) stops the variance from swamping the mean's gradient.
    nll = 0.5 * (logvar + (target - mean) ** 2 / logvar.exp())
    return (nll * logvar.detach().exp() ** BETA_NLL).mean()

def fusion_loss(out, target):
    return (gaussian_nll(out['mu'], out['logvar'], target)
            + LAMBDA * gaussian_nll(out['mu1'], out['lv1'], target)
            + (1 - LAMBDA) * gaussian_nll(out['mu2'], out['lv2'], target))

@torch.no_grad()
def fusion_scores(model, windows):
    # Squared error of the fused mean, the uncertainty-aware score z², and the gate w; each (time, dims).
    model.eval()
    out = model(windows)
    error = (out['mu'] - windows[:, -1]) ** 2
    return error.cpu().numpy(), (error / out['logvar'].exp()).cpu().numpy(), out['w'].cpu().numpy()

print('parameters (25 dims):', sum(p.numel() for p in AdaptiveDTAAD(25).parameters()))

In [ ]:
FUSION_VARIANTS = ['DTAAD (paper)', 'DTAAD + calibrated POT', 'adaptive fusion',
                   'adaptive + uncertainty score', 'adaptive + uncertainty + calibrated POT']

def anomaly_segments(chan):
    segments = []
    for _, row in meta[meta.chan_id == chan].iterrows():
        kinds = [k.strip() for k in row['class'].strip('[]').split(',')]
        segments += [(start, end, kind) for (start, end), kind in zip(ast.literal_eval(row.anomaly_sequences), kinds)]
    return segments

def compare_fusion(ds, chan, seed_index):
    seed = SEEDS[seed_index]
    torch.manual_seed(seed); np.random.seed(seed)
    train_w, test_w, labels = prepare(chan, 1.0, seed)
    n_val = len(train_w) // 5
    model = AdaptiveDTAAD(train_w.shape[2]).to(device)
    train_model(model, train_w[:-n_val], train_w[-n_val:], LR)
    baseline = per_channel_runs[(ds, 1.0, chan)][seed_index]['model']
    splits = (train_w, train_w[-n_val:], test_w)                         # scores for training, validation, test
    base = [anomaly_scores(baseline, w) for w in splits]
    error, z, gate = zip(*[fusion_scores(model, w) for w in splits])
    scores = dict(zip(FUSION_VARIANTS, [(base, False), (base, True), (error, False), (z, False), (z, True)]))
    gate = gate[2].mean(axis=1)                                          # mean global-branch weight per test step
    results, segment_rows = {}, []
    for name, ((train_s, val_s, test_s), calibrated) in scores.items():
        if calibrated:                                                   # validation-fitted, risk q split over sensors
            _, metrics, adjusted, raw = evaluate_dimensions(val_s, test_s, labels, ds, q=POT_Q / test_s.shape[1])
        else:
            _, metrics, adjusted, raw = evaluate_dimensions(train_s, test_s, labels, ds)
        results[name] = dict(metrics=metrics, labels=labels, adjusted=adjusted.any(axis=1))
        raw = raw.any(axis=1)
        for start, end, kind in anomaly_segments(chan):
            hits = np.where(raw[start:end + 1])[0]
            segment_rows.append({'dataset': ds, 'channel': chan, 'seed': seed, 'variant': name, 'type': kind,
                                 'length': 'short (<100)' if end - start + 1 < 100 else 'long (>=100)',
                                 'detected': len(hits) > 0, 'delay': hits[0] if len(hits) else np.nan,
                                 'gate': gate[start:end + 1].mean()})
    return results, segment_rows, gate[labels == 0].mean()

def pooled_detection(results):
    labels = np.concatenate([r['labels'] for r in results])
    adjusted = np.concatenate([r['adjusted'] for r in results])
    p, r, f1 = f1_score(adjusted, labels)
    return {'P': p, 'R': r, 'F1': f1, 'AUC': roc_auc_score(labels, adjusted)}

In [ ]:
fusion_runs = {(ds, chan, i): compare_fusion(ds, chan, i)
               for ds, channels in CHANNELS.items() for chan in channels for i in range(len(SEEDS))}

rows = []
for ds, channels in CHANNELS.items():
    for name in FUSION_VARIANTS:
        seed_metrics = pd.DataFrame([pooled_detection([fusion_runs[(ds, chan, i)][0][name] for chan in channels])
                                     for i in range(len(SEEDS))])
        rows.append({'dataset': ds, 'variant': name, 'P': f'{seed_metrics.P.mean():.3f}', 'R': f'{seed_metrics.R.mean():.3f}',
                     'F1': f'{seed_metrics.F1.mean():.3f} +/- {seed_metrics.F1.std():.3f}', 'AUC': f'{seed_metrics.AUC.mean():.3f}'})
fusion_table = pd.DataFrame(rows)
print(fusion_table.to_string(index=False))

segments = pd.DataFrame([row for _, segment_rows, _ in fusion_runs.values() for row in segment_rows])
by_type = (segments.groupby(['dataset', 'type', 'length', 'variant'], sort=False)
           .agg(segments=('detected', 'size'), detected=('detected', 'mean'), median_delay=('delay', 'median'))
           .reset_index())
by_type['segments'] //= len(SEEDS)
print('\nSegment detection rate and median delay (steps) by anomaly type and length')
print(by_type.round(3).to_string(index=False))

gate_table = segments[segments.variant == FUSION_VARIANTS[0]].groupby(['dataset', 'type']).gate.mean().unstack()
gate_table['normal'] = [np.mean([g for (d, _, _), (_, _, g) in fusion_runs.items() if d == ds]) for ds in gate_table.index]
print('\nMean weight w of the global branch')
print(gate_table.round(3).to_string())

fusion_table.to_csv('results/fusion_results.csv', index=False)
by_type.to_csv('results/fusion_by_type.csv', index=False)
gate_table.to_csv('results/fusion_gate.csv')